In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. 패키지

In [2]:
# python-dotenv, langchain, langchain-openai, langchain-pinecone, langchin-community, docx2txt
# langchain-upstage, langchain-ollama
# pandas

# 2. 환경변수 설정

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-pro4"
# OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4096

PINECONE_INDEX_NAME = "better-rag-index"
PINECONE_INDEX_METRIC = 'cosine'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_DIMENSION = 4096

# 3. 문서를 chunk로 분할하기

In [4]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('tax_docx/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 1500글자 기준으로 분리
    chunk_overlap=100, 
    separators=['\n\n', '\n', ' ', '']
)
# documents = loader.load_and_split(text_splitter=text_splitter)
document = loader.load()
documents = text_splitter.split_documents(document)
print(f"총 {len(documents)}개 청크로 분리")

C:\Users\EZ\AppData\Local\Temp\ipykernel_9032\606412924.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


총 184개 청크로 분리


# 4. metadata 추가하기
- chunk 내용의 title, category, 조항

## title추출

In [5]:
import re
def remove_special_chars(text:str) -> str:
    "특수문자 및 \n제거 (:, .는 그대로)"
    text = text.replace("\n", " ")
    # 한글, 영문, 숫자, 공백, . , : |-만 남기고 전부 제거
    cleaned = re.sub(r"[^가-힣a-zA-Z0-9\s\.,:|\-]", "", text).strip()
    # 불필요한 중복 공백을 제거
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned
content = " ▶️ **소득세 납세 의무 ** : \n\n|이 법 또는 「법인세법」에 따라 소득에 대한 소득세를 부과  "
remove_special_chars(content)

'소득세 납세 의무 : |이 법 또는 법인세법에 따라 소득에 대한 소득세를 부과'

In [6]:
remove_special_chars(documents[43].page_content)

'전문개정 2009. 12. 31. 제목개정 2014. 1. 1. 제54조의2공동사업에 대한 소득공제 등 특례 제51조의3 또는 조세특례제한법에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출납입투자출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출납입투자출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. 개정 2012. 1. 1., 2014. 1. 1. 전문개정 2009. 12. 31. 제목개정 2014. 1. 1. 제4절 세액의 계산 개정 2009. 12. 31. 제1관 세율 개정 2009. 12. 31. 제55조세율 거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액이하 종합소득산출세액이라 한다을 그 세액으로 한다. 개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31. | 종합소득 과세표준 | 세율 | |---|---| | 1,400만원 이하 | 과세표준의 6퍼센트 | | 1,400만원 초과 5,000만원 이하 | 84만원 1,400만원을 초과하는 금액의 15퍼센트 | | 5,000만원 초과 8,800만원 이하 | 624만원 5,000만원을 초과하는 금액의 24퍼센트 | | 8,800만원 초과 1억5천만원 이하 | 1,536만원 8,800만원을 초과하는 금액의 35퍼센트 | | 1억5천만원 초과 3억원 이하 | 3,706만원 1억5천만원을 초과하는 금액의 38퍼센트 | | 3억원 초과 5억원 이하 | 9,406만원 3억원을 초과하는 금액의 40퍼센트 | | 5억원 초과 10억원 이하 | 1억7,406만원 5억원을 초과하는 금액의 42퍼센트 | | 10억원 초과 | 

In [7]:
%%time
# 제목을 추출하는 함수(사용 llm은 exaone3.5:2.4b) ollama list -> ollama pull exaone3.5:2.4b
from langchain_ollama import ChatOllama
from langchain_upstage import ChatUpstage
def extract_title_with_llm(content):
    "exaone 3.5를 사용하여 content의 제목을 반환"
    title_extractor_llm = ChatOllama(
        model = 'exaone3.5:2.4b',
        temperature=0.1
    )
#     title_extractor_llm = ChatUpstage(
#         model = UPSTAGE_LLM_MODEL,
#         temperature=0.1
#     )
    prompt = f"""다음 소득세법 조문의 핵심 제목을 100토큰 이내로 완벽하게 말이 되도록 추출해 주세요.
    중간에 말이 끊기면 안 되요.
    조문 : {content}"""
    ai_message = title_extractor_llm.invoke(prompt)
    title = ai_message.content.strip()
    return remove_special_chars(title)
extract_title_with_llm(remove_special_chars(documents[43].page_content))

CPU times: total: 2.64 s
Wall time: 3.63 s


'공동사업 소득공제 특례 및 종합소득퇴직소득 세율과 세액공제'

## 카테고리 분류

In [8]:
def categorize_content(content):
    "content의 카테고리를 list로 반환"
    # gpt가 meta데이터 활용한다고 하니 만들어준 카테고리
    categories = {
    '납세의무': ['납세의무', '거주자', '비거주자', '원천징수', '공동사업자', '상속인', '증여자', '신탁재산'],
    '세율계산': ['세율', '과세표준', '산출세액', '결정세액', '종합소득', '퇴직소득'],
    '근로소득': ['근로소득', '총급여', '급여', '연봉', '임금', '퇴직소득'],
    '사업소득': ['사업소득', '공동사업', '주택임대소득', '부업소득'],
    '이자배당': ['이자소득', '배당소득', '예금이자', '채권', '의제배당'],
    '양도소득': ['양도소득', '자산양도', '부동산양도', '주식양도'],
    '연금소득': ['연금소득', '공적연금', '사적연금', '연금보험'],
    '기타소득': ['기타소득', '상금', '보상금', '발명보상금', '종교인소득'],
    '공제감면': ['공제', '소득공제', '세액공제', '기본공제', '감면'],
    '비과세': ['비과세', '면제', '복무급여', '실업급여', '출산휴가급여', '장학금'],
    '신고납부': ['신고', '납부', '납세지', '신고기한', '원천징수'],
    '과세기간': ['과세기간', '과세연도', '사업연도'],
    '과세소득구분': ['종합소득', '퇴직소득', '양도소득', '금융투자소득'],
    }
    result = [] # 각 카테고리를 추가
    for category, keywords in categories.items():
        #print(category, keywords)
        # print(any([keyword in content for keyword in keywords])) : 하나라도 True인지 여부
        if any([keyword in content for keyword in keywords]):
            result.append(category)
    if not result:
        result.append("기타")
    return result

In [9]:
print(categorize_content(documents[43].page_content))
print(categorize_content("아무 내용도 없다"))

['납세의무', '세율계산', '근로소득', '사업소득', '공제감면', '과세기간', '과세소득구분']
['기타']


- 개선된 카테고리 분류방법 : 키워드의 중요도에 따라 가중치를 부여하고, 가중치가 높은 순으로 카테고리를 반환

In [10]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },

        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },

        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },

        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },

        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },

        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },

        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },

        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },

        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },

        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },

        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },

        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },

        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },

        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }


In [11]:
def categorize_content(content, top_k=None):
    """
    content의 카테고리를 분류 - 점수 기반으로 모든 카테고리를 점수순으로 list를 반환(top_k만큼)
        Parameters :
        - content : 카테고리를 분류할 텍스트
        - top_k : 상위 몇개까지의 카테고리를 반환할지 정수나 None(None이면 모든 카테고리를 반환)
        Returns :
        - 카테고리 리스트(점수순) ex.{'장부기장':30, '과세방식':45}=>['과세방식','장부기장']
    """
    category_keywords = get_category()
    category_scores = {}
    # 각 카테고리별 점수 계산
    for category, weighted_keywords in category_keywords.items():
        # print(category, weighted_keywords)
        score = 0
        for keyword, weight in weighted_keywords.items():
            # print(keyword, weight)
            count = content.count(keyword) # content에 keyword가 나오는 횟수
            score += count * weight
        if score>0:
            category_scores[category] = score
    # print(category_scores)
    sorted_categories = sorted(category_scores.items(), key=lambda item:item[1], reverse=True)#내림차순
    # print(sorted_categories)
    all_categories = [category[0] for category in sorted_categories]
    # 매칭되는 카테고리가 없을 경우 ["기타"]를 반환
    if not all_categories:
        all_categories = ["기타"]
    # print(all_categories)
    if top_k is not None: #if top_k:
        return all_categories[:top_k]
    else:
        return all_categories

In [12]:
content = """1. 원천징수하는 자가 거주자인 경우: 그 거주자의 주된 사업장 소재지. 다만, 주된 사업장 외의 사업장에서 원천징수를 하는 경우에는 그 사업장의 소재지, 사업장이 없는 경우에는 그 거주자의 주소지 또는 거소지로 한다.
2. 원천징수하는 자가 비거주자인 경우: 그 비거주자의 주된 국내사업장 소재지. 다만, 주된 국내사업장 외의 국내사업장에서 원천징수를 하는 경우에는 그 국내사업장의 소재지, 국내사업장이 없는 경우에는 그 비거주자의 거류지(居留地) 또는 체류지로 한다.
3. 원천징수하는 자가 법인인 경우: 그 법인의 본점 또는 주사무소의 소재지
장부 장부 장부 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기
"""
#content = "카테고리의 아무 단어도 없네"
print(categorize_content(content))
print(categorize_content("연봉 5천만원인 직장인의 소득세는 얼마예요?"))
print(categorize_content(documents[43].page_content))

['장부기장', '납세의무']
['근로소득', '세율계산']
['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간']


## 조항 추출하기

In [13]:
# content에서 제000조 스타일만 추출 : "「9조,60조」"
import re
article_match = re.findall(r"제(\d{1,3}조)", "제60조 뭐라뭐라 제60조 뭐라뭐라 제9조 어쩌구저쩌구")
article = list(set(article_match))
# article = sorted(article, key=lambda x : int(x[:-1]))
article.sort(key=lambda x : int(x[:-1]))
"「" + ", ".join(article) + "」"

'「9조, 60조」'

In [14]:
a = '60조'
int(a[:-1])

60

In [15]:
import re
def get_article(content):
    """
    content내의 조항들을 반환
    Parameter :
    - content : 조항을 추출한 text
    Returns : content내에 등장된 조항 수들 문자로 반환 ex."「9조,60조」"
    """
    article_match = re.findall(r"제(\d{1,3}조)", content)
    article_list = list(set(article_match))
    article_list.sort(key=lambda x : int(x[:-1]))
    if article_list:
        return "「" + ", ".join(article_list) + "」"
    return ""

In [16]:
print(get_article(content))
print(get_article(documents[43].page_content))


「43조, 51조, 54조, 55조, 59조」


In [17]:
# documents
metadata={'source': 'tax_docx/소득세법_with_markdown.docx'}
metadata['title'] = '제목'
metadata['category'] = ['납세의무','소득공제']
metadata['article'] = '「43조, 51조, 54조, 55조, 59조」'
metadata

{'source': 'tax_docx/소득세법_with_markdown.docx',
 'title': '제목',
 'category': ['납세의무', '소득공제'],
 'article': '「43조, 51조, 54조, 55조, 59조」'}

In [18]:
documents[0].metadata

{'source': 'tax_docx/소득세법_with_markdown.docx'}

In [19]:
type(documents[0])

langchain_core.documents.base.Document

In [20]:
%%time
# documents의 metadata추가
from langchain_core.documents import Document
enhanced_chunks = [] # metadata를 추가한 청크들(documents)
for i, chunk in enumerate(documents[:15]):
    if i%5==0:
        print(f"진행 : {i:2} / 15 : {i/15*100:.1f}")
    
    content = chunk.page_content # str
    metadata = chunk.metadata.copy()    # dict
    metadata['title'] = extract_title_with_llm(remove_special_chars(content)) # 제목 추출
    metadata['category'] = categorize_content(content) # 카테고리 추출
    metadata['chunk_id'] = f'chunk_{i:03d}'
    # 조문 추출 : "" + "「0조」"
    article = get_article(content)
    if article:
        metadata['article'] = article # 추출된 조항이 있을 경우만 metadata에 추가
    enhanced_chunks.append(Document(page_content=content,
                                   metadata=metadata)) 
print(f"★ {len(enhanced_chunks)}개 청크 전처리 완료")

진행 :  0 / 15 : 0.0
진행 :  5 / 15 : 33.3
진행 : 10 / 15 : 66.7
★ 15개 청크 전처리 완료
CPU times: total: 156 ms
Wall time: 14.4 s


In [24]:
type(enhanced_chunks[0].metadata['category'])

list

In [34]:
import pandas as pd
# enhanced_chunks -> 딕셔너리리스트 -> 데이터프레임으로 변환(csv백업)
data = []
for chunk in enhanced_chunks:
    row = {"page_content":chunk.page_content}
    row.update(chunk.metadata)
    data.append(row)
df = pd.DataFrame(data)
df.to_csv("./tax_docx/소득세법enhanced_chunks일부.csv", index=False, encoding='cp949')

## 백업받은 csv를 읽어오기

In [38]:
import pandas as pd
df = pd.read_csv("./tax_docx/소득세법enhanced_chunks.csv", encoding='cp949')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page_content  184 non-null    object
 1   source        184 non-null    object
 2   title         184 non-null    object
 3   category      184 non-null    object
 4   chunk_id      184 non-null    object
 5   article       181 non-null    object
dtypes: object(6)
memory usage: 8.8+ KB


In [39]:
df.head(2)

,page_content,source,title,category,chunk_id,article
0,소득세법\n\n\n\n소득세법\n\n[시행 2026. 7. 1.] [법률 제2122...,tax_docx/소득세법_with_markdown.docx,소득세법 총칙 개인 소득에 따라 적정 과세하여 조세부담 형평과 재정수입 확보를 목적...,"['납세의무', '세율계산', '신고납부', '사업소득', '이자배당', '근로소득...",chunk_000,"「1조, 2조, 13조」"
1,1. 구성원 간 이익의 분배비율이 정하여져 있고 해당 구성원별로 이익의 분배비율이 ...,tax_docx/소득세법_with_markdown.docx,소득세 및 법인세 납세의무 분배 - 구성원별 분배 확인 시: 구성원별로 소득세 또는...,"['납세의무', '세율계산', '사업소득', '양도소득', '신고납부', '과세방식']",chunk_001,"「1조, 2조, 14조, 43조, 44조, 101조, 119조, 127조」"


In [40]:
df.loc[0, 'category']

"['납세의무', '세율계산', '신고납부', '사업소득', '이자배당', '근로소득', '양도소득', '기타소득']"

In [42]:
# 문자열("['납세의무', '세율계산']") => 리스트(['납세의무', '세율계산'])
import ast
category = "['납세의무', '세율계산']"
# 리스트로 변환
category_lst = ast.literal_eval(category)
print(category_lst, type(category_lst))

['납세의무', '세율계산'] <class 'list'>


In [49]:
# df['category'] 타입변경(문자열->리스트)
df['category'] = df['category'].apply(ast.literal_eval)
df.loc[0, 'category']

['납세의무', '세율계산', '신고납부', '사업소득', '이자배당', '근로소득', '양도소득', '기타소득']

## 데이터프레임 -> Document 리스트 -> pinecone 저장

In [60]:
for i, row in df.loc[0:1].iterrows():
    print(i)
    print(row)
    print(type(row))

0
page_content    소득세법\n\n\n\n소득세법\n\n[시행 2026. 7. 1.] [법률 제2122...
source                           tax_docx/소득세법_with_markdown.docx
title           소득세법 총칙 개인 소득에 따라 적정 과세하여 조세부담 형평과 재정수입 확보를 목적...
category         [납세의무, 세율계산, 신고납부, 사업소득, 이자배당, 근로소득, 양도소득, 기타소득]
chunk_id                                                chunk_000
article                                             「1조, 2조, 13조」
Name: 0, dtype: object
<class 'pandas.core.series.Series'>
1
page_content    1. 구성원 간 이익의 분배비율이 정하여져 있고 해당 구성원별로 이익의 분배비율이 ...
source                           tax_docx/소득세법_with_markdown.docx
title           소득세 및 법인세 납세의무 분배 - 구성원별 분배 확인 시: 구성원별로 소득세 또는...
category                     [납세의무, 세율계산, 사업소득, 양도소득, 신고납부, 과세방식]
chunk_id                                                chunk_001
article                 「1조, 2조, 14조, 43조, 44조, 101조, 119조, 127조」
Name: 1, dtype: object
<class 'pandas.core.series.Series'>


In [62]:
%%time
from langchain_core.documents import Document
import pandas as pd
enhanced_chunks = []
for i, row in df.iterrows():
    content = row['page_content']
    metadata = {
        'source': row['source'], # row.source
        'title':row['title'],
        'category':row['category'],
        'chunk_id' : row['chunk_id'],
    }
    # article은 결측치가 아닐 경우만 추가
    if pd.notna(row['article']):
        metadata['article'] = row['article']
    print(metadata)

{'source': 'tax_docx/소득세법_with_markdown.docx', 'title': '소득세법 총칙 개인 소득에 따라 적정 과세하여 조세부담 형평과 재정수입 확보를 목적으로 함. 거주자 및 비거주자 구분 과세 의무 부여.', 'category': ['납세의무', '세율계산', '신고납부', '사업소득', '이자배당', '근로소득', '양도소득', '기타소득'], 'chunk_id': 'chunk_000', 'article': '「1조, 2조, 13조」'}
{'source': 'tax_docx/소득세법_with_markdown.docx', 'title': '소득세 및 법인세 납세의무 분배 - 구성원별 분배 확인 시: 구성원별로 소득세 또는 법인세 납부 의무 발생. - 무분별 분배 시: 전체 구성원 중 일부만 확인되면, 그에 따라 납세 의무 분담. - 비거주자 및 법인 아닌 단체: 국외 투자기구의 경우, 국내 원천소득 귀속 시 비거주자로 간주되어 소득세 납부 의무. - 공동사업자: 소득 합산 시 주된 공동사업자와 특수관계인 연대 납세 의무. - 상속 및 증여: 피상속인 및 증여자의 소득에 대한 납세 의무 분담. - 원천징수 소득: 합산되지 않는 원천징수 소득에 대한 납세 의무. - 공동 소유 자산: 양도소득금액 계산 시 공동 소유자별 납세 의무.', 'category': ['납세의무', '세율계산', '사업소득', '양도소득', '신고납부', '과세방식'], 'chunk_id': 'chunk_001', 'article': '「1조, 2조, 14조, 43조, 44조, 101조, 119조, 127조」'}
{'source': 'tax_docx/소득세법_with_markdown.docx', 'title': '소득세법 주요 조항 요약 - 신탁재산 귀속 소득: 신탁 이익은 수익자 사망 시 상속인에게, 실질 통제 시 위탁자에게 귀속. - 과세 대상: 거주자는 모든 소득 과세, 외국인 거주자는 국내 발생 국외 소득 일부 과세, 비거주자는 국내 원천소득 과세. -

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page_content  184 non-null    object
 1   source        184 non-null    object
 2   title         184 non-null    object
 3   category      184 non-null    object
 4   chunk_id      184 non-null    object
 5   article       181 non-null    object
dtypes: object(6)
memory usage: 8.8+ KB
